# Pydantic e tipos `Annotated`

O Python oferece um mecanismo para anexar metadados a qualquer tipo.

In [1]:
from typing import Annotated

Definimos um tipo anotado assim:

In [2]:
SpecialInt = Annotated[int, "metadata 1", [1, 2, 3], 100]

Para o Python, não há nada de especial nesse tipo anotado. Não é um tipo novo — continua sendo `int`, só que com informação extra anexada.

Com Python, podemos recuperar esses metadados se quisermos usá-los para algo.

In [3]:
from typing import get_args

In [4]:
print(f"get_args(SpecialInt): {get_args(SpecialInt)!r}")

get_args(SpecialInt): (<class 'int'>, 'metadata 1', [1, 2, 3], 100)


O Pydantic sabe aproveitar essas anotações.

Elas precisam ser anotações que o Pydantic entenda (ou seja, objetos específicos). Isso dá uma forma muito flexível de acrescentar
funcionalidade aos tipos: restrições, validadores e muito mais.

Comecemos com um exemplo simples.

In [5]:
from pydantic import BaseModel, Field

In [6]:
class Model(BaseModel):
    x: int = Field(gt=0, le=100)
    y: int = Field(gt=0, le=100)
    z: int = Field(gt=0, le=100)

Repare que `x`, `y` e `z` são, no fundo, o mesmo tipo.

In [7]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'x': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0), Le(le=100)]), 'y': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0), Le(le=100)]), 'z': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0), Le(le=100)])}


Vamos usar anotações para definir um int anotado que inclui a definição do `Field` nas anotações.

In [8]:
BoundedInt = Annotated[int, Field(gt=0, le=100)]

Para o Python, é só um `int` com dados extras. Mas o Pydantic sabe tratar o objeto `Field` dentro das anotações.

In [9]:
class Model(BaseModel):
    x: BoundedInt
    y: BoundedInt
    z: BoundedInt

In [10]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'x': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0), Le(le=100)]), 'y': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0), Le(le=100)]), 'z': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0), Le(le=100)])}


As definições dos campos, usando anotações, são idênticas às de antes.

E a validação acontece como esperado:

In [11]:
print(f"Model(x=10, y=20, z=30): {Model(x=10, y=20, z=30)!r}")

Model(x=10, y=20, z=30): Model(x=10, y=20, z=30)


In [12]:
try:
    Model(x=0, y=10, z=103)
except ValueError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
x
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
z
  Input should be less than or equal to 100 [type=less_than_equal, input_value=103, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


(O `except ValueError` funciona porque `ValidationError` é subclasse de `ValueError`.)

Tipos anotados economizam muita digitação e ajudam a manter o código DRY (*don't repeat yourself*).

Também dá para usar tipos anotados direto no modelo — não é obrigatório defini-los antes.

In [13]:
class Model(BaseModel):
    field_1: Annotated[int, Field(gt=0)] = 1
    field_2: Annotated[str, Field(min_length=1, max_length=10)] | None = None

In [14]:
print(f"Model(): {Model()!r}")

Model(): Model(field_1=1, field_2=None)


In [15]:
print(f"Model(field_1=10): {Model(field_1=10)!r}")

Model(field_1=10): Model(field_1=10, field_2=None)


In [16]:
print(f'Model(field_2="Python"): {Model(field_2="Python")!r}')

Model(field_2="Python"): Model(field_1=1, field_2='Python')


E a validação funciona como esperado:

In [17]:
try:
    Model(field_1=-10, field_2 = "Python" * 3)
except ValueError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
field_1
  Input should be greater than 0 [type=greater_than, input_value=-10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
field_2
  String should have at most 10 characters [type=string_too_long, input_value='PythonPythonPython', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_long


Voltaremos a usar anotações com validadores e serializadores em breve.